In [ ]:
import os
import time
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, VGAE
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import pandas as pd
import datetime
from sklearn.model_selection import KFold

if torch.cuda.is_available():
    print("GPU is available")
    print(f"Using device: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is not available, using CPU")
    
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# Data paths
standardized_data_folder = "/home/nishioka/GNN/Defect_4x4_Normalized1"
label_data_folder = "/home/nishioka/GNN/Defectlabel4x4/DefectLabels_4x4_test1"

# Load coordinate data
x_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/x_2layer_normalized.npy")[:3654]
y_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/y_2layer_normalized.npy")[:3654]
z_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/z_2layer_normalized.npy")[:3654]

# Load edge information
edges = np.load("/home/nishioka/GNN/BasicdataforGNN/edges_2layer.npy")
edge_index = torch.tensor(edges.T, dtype=torch.long).to(device)

# Data and label files
data_files = [f for f in os.listdir(standardized_data_folder) if f.startswith("Normalized1_Defect4x4_ELNOD")]
label_files = [f for f in os.listdir(label_data_folder) if f.startswith("DefectLabel_L")]

# Function to extract layer and block information from filenames
def extract_layer_block(file_name):
    if file_name.startswith("0"):
        return (0, 0)  # No defect data
    try:
        layer_block_str = file_name.split("_")[-1].replace(".npy", "")
        layer_str = layer_block_str.split("L")[1].split("B")[0]
        block_str = layer_block_str.split("B")[1]
        layer = int(layer_str)
        block = int(block_str)
        return (layer, block)
    except (ValueError, IndexError):
        print(f"Invalid file name format: {file_name}")
        return None

# Prepare data pairs
data_label_pairs = {}
for data_file in data_files:
    layer_block = extract_layer_block(data_file)
    if layer_block:
        data_label_pairs[layer_block] = {"data": data_file}

for label_file in label_files:
    layer_block = extract_layer_block(label_file)
    if layer_block and layer_block in data_label_pairs:
        data_label_pairs[layer_block]["label"] = label_file

# Valid pairs and adding defect-free data
valid_pairs = [(v["data"], v["label"]) for k, v in data_label_pairs.items() if "label" in v]
defect_free_pair = ("/home/nishioka/GNN/BasicdataforGNN/Normalized1_nodefect_ElNOD.npy",
                    "/home/nishioka/GNN/BasicdataforGNN/DefectLabel_nodefect.npy")
for _ in range(100):
    valid_pairs.append(defect_free_pair)

# Prepare data
def prepare_data(pairs):
    sampled_data = []
    sampled_labels = []
    
    for data_file, label_file in pairs:
        data_file_path = os.path.join(standardized_data_folder, data_file)
        label_file_path = os.path.join(label_data_folder, label_file)

        # Load data and labels
        try:
            values = np.load(data_file_path)[:3654]
            label = np.load(label_file_path)[:3654]
        except Exception as e:
            print(f"Error loading data: {e}")
            continue
        
        # Create node features by combining coordinate and stress data
        node_features = np.vstack((x_coords, y_coords, z_coords, values)).T
        sampled_data.append(node_features)
        sampled_labels.append(label)
    
    if len(sampled_data) == 0 or len(sampled_labels) == 0:
        print("No valid data found in the pairs.")
        return None

    # Convert to tensor
    sampled_data = np.stack(sampled_data)
    sampled_labels = np.hstack(sampled_labels)

    x = torch.tensor(sampled_data.reshape(-1, 4), dtype=torch.float).to(device)
    y = torch.tensor(sampled_labels, dtype=torch.float).to(device)

    return x, y

all_x, all_y = prepare_data(valid_pairs)

# Create Data objects for each sample
all_data_list = []
num_all_samples = all_x.shape[0] // 3654
for i in range(num_all_samples):
    all_data_list.append(Data(x=all_x[i * 3654:(i + 1) * 3654], edge_index=edge_index, y=all_y[i * 3654:(i + 1) * 3654]))

# Define GVAE model class
class Encoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(Encoder, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels * 2)
        self.conv_mu = GCNConv(hidden_channels * 2, hidden_channels)
        self.conv_logstd = GCNConv(hidden_channels * 2, hidden_channels)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        return self.conv_mu(x, edge_index), self.conv_logstd(x, edge_index)

class Decoder(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super(Decoder, self).__init__()
        self.linear = nn.Linear(hidden_channels, out_channels)

    def forward(self, z):
        return self.linear(z)

# Manual grid search for hyperparameter tuning
hidden_channels_list = [32, 64, 128]
learning_rate_list = [1e-4, 1e-3, 1e-2]
weight_decay_list = [1e-5, 1e-4, 1e-3]
epochs_list = [100, 200, 300]

best_loss = float('inf')
best_config = None

# Grid search over all combinations of hyperparameters
for hidden_channels in hidden_channels_list:
    for learning_rate in learning_rate_list:
        for weight_decay in weight_decay_list:
            for epochs in epochs_list:
                print(f"Training with hidden_channels={hidden_channels}, learning_rate={learning_rate}, weight_decay={weight_decay}, epochs={epochs}")
                
                encoder = Encoder(in_channels=4, hidden_channels=hidden_channels).to(device)
                decoder = Decoder(hidden_channels=hidden_channels, out_channels=4).to(device)
                model = VGAE(encoder).to(device)
                optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=learning_rate, weight_decay=weight_decay)

                def loss_fn(recon_x, x, mu, logstd):
                    recon_loss = F.mse_loss(recon_x, x, reduction='mean')
                    kl_divergence = -0.5 * torch.mean(1 + 2 * logstd - mu.pow(2) - (2 * logstd).exp())
                    return recon_loss + kl_divergence

                # Training loop
                for epoch in range(1, epochs + 1):
                    model.train()
                    optimizer.zero_grad()
                    
                    total_loss = 0
                    for data in all_data_list:
                        data = data.to(device)
                        x = data.x.to(device)
                        edge_index = data.edge_index.to(device)
                        mu, logstd = encoder(data.x, data.edge_index)
                        z = mu + torch.exp(logstd) * torch.randn_like(mu)
                        recon = decoder(z.to(device))  # Ensure z is on the same device as the decoder
                        loss = loss_fn(recon, x, mu, logstd)
                        loss.backward()
                        optimizer.step()
                        total_loss += loss.item()

                    avg_loss = total_loss / len(all_data_list)
                    if epoch % 10 == 0:
                        print(f"Epoch {epoch}/{epochs}, Loss: {avg_loss:.4f}")

                # Update best configuration if current loss is lower
                if avg_loss < best_loss:
                    best_loss = avg_loss
                    best_config = {
                        "hidden_channels": hidden_channels,
                        "learning_rate": learning_rate,
                        "weight_decay": weight_decay,
                        "epochs": epochs
                    }

print("Best hyperparameters found were: ", best_config)

# Train final model with best hyperparameters
hidden_channels = best_config["hidden_channels"]
learning_rate = best_config["learning_rate"]
weight_decay = best_config["weight_decay"]
epochs = best_config["epochs"]

encoder = Encoder(in_channels=4, hidden_channels=hidden_channels)
decoder = Decoder(hidden_channels=hidden_channels, out_channels=4)
model = VGAE(encoder).to(device)
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=learning_rate, weight_decay=weight_decay)

for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad()
    
    total_loss = 0
    for data in all_data_list:
        data = data.to(device)
        mu, logstd = encoder(data.x, data.edge_index)
        z = model.reparameterize(mu, logstd)
        recon = decoder(z)  # Use the decoder to reconstruct the input
        loss = loss_fn(recon, data.x, mu, logstd)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(all_data_list)
    if epoch % 10 == 0:
        print(f"Epoch {epoch}/{epochs}, Loss: {avg_loss:.4f}")

# Save the trained model
torch.save(model.state_dict(), f'/home/nishioka/GNN/GNNmodel/gvae_model_{timestamp}.pth')

# Plot training loss (for simplicity, a placeholder example)
plt.figure()
plt.plot(range(1, epochs + 1), [avg_loss] * epochs)  # Placeholder; replace with actual epoch losses
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('GVAE Training Loss')
plt.savefig(f'/home/nishioka/GNN/GNNmodelgraphfig/gvae_loss_{timestamp}.png')
plt.show()

GPU is available
Using device: NVIDIA GeForce RTX 3090
Training with hidden_channels=32, learning_rate=0.0001, weight_decay=1e-05, epochs=100
Epoch 10/100, Loss: 0.0643
Epoch 20/100, Loss: 0.0639
Epoch 30/100, Loss: 0.0631
Epoch 40/100, Loss: 0.0621
Epoch 50/100, Loss: 0.0629
Epoch 60/100, Loss: 0.0620
Epoch 70/100, Loss: 0.0625
Epoch 80/100, Loss: 0.0627
Epoch 90/100, Loss: 0.0632
Epoch 100/100, Loss: 0.0620
Training with hidden_channels=32, learning_rate=0.0001, weight_decay=1e-05, epochs=200
Epoch 10/200, Loss: 0.0646
Epoch 20/200, Loss: 0.0624
Epoch 30/200, Loss: 0.0633
Epoch 40/200, Loss: 0.0624
Epoch 50/200, Loss: 0.0632
Epoch 60/200, Loss: 0.0620
Epoch 70/200, Loss: 0.0625
Epoch 80/200, Loss: 0.0631
Epoch 90/200, Loss: 0.0627
Epoch 100/200, Loss: 0.0625
Epoch 110/200, Loss: 0.0628
Epoch 120/200, Loss: 0.0625
Epoch 130/200, Loss: 0.0624
Epoch 140/200, Loss: 0.0623
Epoch 150/200, Loss: 0.0629
Epoch 160/200, Loss: 0.0622
Epoch 170/200, Loss: 0.0625
Epoch 180/200, Loss: 0.0626
Epoch

In [1]:
import torch
torch.save(model.state_dict(), f'/home/nishioka/GNN/GNNmodel/gvae_model_{timestamp}.pth')

NameError: name 'model' is not defined

In [2]:
import matplotlib.pyplot as plt
# Plot training loss (for simplicity, a placeholder example)
plt.figure()
plt.plot(range(1, epochs + 1), [avg_loss] * epochs)  # Placeholder; replace with actual epoch losses
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('GVAE Training Loss')
plt.savefig(f'/home/nishioka/GNN/GNNmodelgraphfig/gvae_loss_{timestamp}.png')
plt.show()

NameError: name 'epochs' is not defined

<Figure size 640x480 with 0 Axes>

In [3]:
import os
import time
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.nn import GCNConv, VGAE
from torch_geometric.data import Data
from torch_geometric.loader import DataLoader
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
import matplotlib.pyplot as plt
import pandas as pd
import datetime
from sklearn.model_selection import KFold

if torch.cuda.is_available():
    print("GPU is available")
    print(f"Using device: {torch.cuda.get_device_name(0)}")
else:
    print("GPU is not available, using CPU")
    
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
timestamp = datetime.datetime.now().strftime("%Y%m%d_%H%M%S")

# Data paths
standardized_data_folder = "/home/nishioka/GNN/Defect_4x4_Normalized1"
label_data_folder = "/home/nishioka/GNN/Defectlabel4x4/DefectLabels_4x4_test1"

# Load coordinate data
x_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/x_2layer_normalized.npy")[:3654]
y_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/y_2layer_normalized.npy")[:3654]
z_coords = np.load("/home/nishioka/GNN/BasicdataforGNN/z_2layer_normalized.npy")[:3654]

# Load edge information
edges = np.load("/home/nishioka/GNN/BasicdataforGNN/edges_2layer.npy")
edge_index = torch.tensor(edges.T, dtype=torch.long).to(device)

# Data and label files
data_files = [f for f in os.listdir(standardized_data_folder) if f.startswith("Normalized1_Defect4x4_ELNOD")]
label_files = [f for f in os.listdir(label_data_folder) if f.startswith("DefectLabel_L")]

# Function to extract layer and block information from filenames
def extract_layer_block(file_name):
    if file_name.startswith("0"):
        return (0, 0)  # No defect data
    try:
        layer_block_str = file_name.split("_")[-1].replace(".npy", "")
        layer_str = layer_block_str.split("L")[1].split("B")[0]
        block_str = layer_block_str.split("B")[1]
        layer = int(layer_str)
        block = int(block_str)
        return (layer, block)
    except (ValueError, IndexError):
        print(f"Invalid file name format: {file_name}")
        return None

# Prepare data pairs
data_label_pairs = {}
for data_file in data_files:
    layer_block = extract_layer_block(data_file)
    if layer_block:
        data_label_pairs[layer_block] = {"data": data_file}

for label_file in label_files:
    layer_block = extract_layer_block(label_file)
    if layer_block and layer_block in data_label_pairs:
        data_label_pairs[layer_block]["label"] = label_file

# Valid pairs and adding defect-free data
valid_pairs = [(v["data"], v["label"]) for k, v in data_label_pairs.items() if "label" in v]
defect_free_pair = ("/home/nishioka/GNN/BasicdataforGNN/Normalized1_nodefect_ElNOD.npy",
                    "/home/nishioka/GNN/BasicdataforGNN/DefectLabel_nodefect.npy")
for _ in range(100):
    valid_pairs.append(defect_free_pair)

# Prepare data
def prepare_data(pairs):
    sampled_data = []
    sampled_labels = []
    
    for data_file, label_file in pairs:
        data_file_path = os.path.join(standardized_data_folder, data_file)
        label_file_path = os.path.join(label_data_folder, label_file)

        # Load data and labels
        try:
            values = np.load(data_file_path)[:3654]
            label = np.load(label_file_path)[:3654]
        except Exception as e:
            print(f"Error loading data: {e}")
            continue
        
        # Create node features by combining coordinate and stress data
        node_features = np.vstack((x_coords, y_coords, z_coords, values)).T
        sampled_data.append(node_features)
        sampled_labels.append(label)
    
    if len(sampled_data) == 0 or len(sampled_labels) == 0:
        print("No valid data found in the pairs.")
        return None

    # Convert to tensor
    sampled_data = np.stack(sampled_data)
    sampled_labels = np.hstack(sampled_labels)

    x = torch.tensor(sampled_data.reshape(-1, 4), dtype=torch.float).to(device)
    y = torch.tensor(sampled_labels, dtype=torch.float).to(device)

    return x, y

all_x, all_y = prepare_data(valid_pairs)

# Create Data objects for each sample
all_data_list = []
num_all_samples = all_x.shape[0] // 3654
for i in range(num_all_samples):
    all_data_list.append(Data(x=all_x[i * 3654:(i + 1) * 3654], edge_index=edge_index, y=all_y[i * 3654:(i + 1) * 3654]))

# Define GVAE model class
class Encoder(torch.nn.Module):
    def __init__(self, in_channels, hidden_channels):
        super(Encoder, self).__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, hidden_channels * 2)
        self.conv_mu = GCNConv(hidden_channels * 2, hidden_channels)
        self.conv_logstd = GCNConv(hidden_channels * 2, hidden_channels)

    def forward(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        x = F.relu(self.conv2(x, edge_index))
        return self.conv_mu(x, edge_index), self.conv_logstd(x, edge_index)

class Decoder(torch.nn.Module):
    def __init__(self, hidden_channels, out_channels):
        super(Decoder, self).__init__()
        self.linear = nn.Linear(hidden_channels, out_channels)

    def forward(self, z):
        return self.linear(z)

# Manual grid search for hyperparameter tuning
hidden_channels_list = [128]
learning_rate_list = [1e-4]
weight_decay_list = [1e-5]
epochs_list = [300]

best_loss = float('inf')
best_config = None

# Grid search over all combinations of hyperparameters
for hidden_channels in hidden_channels_list:
    for learning_rate in learning_rate_list:
        for weight_decay in weight_decay_list:
            for epochs in epochs_list:
                print(f"Training with hidden_channels={hidden_channels}, learning_rate={learning_rate}, weight_decay={weight_decay}, epochs={epochs}")
                
                encoder = Encoder(in_channels=4, hidden_channels=hidden_channels).to(device)
                decoder = Decoder(hidden_channels=hidden_channels, out_channels=4).to(device)
                model = VGAE(encoder).to(device)
                optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=learning_rate, weight_decay=weight_decay)

                def loss_fn(recon_x, x, mu, logstd):
                    recon_loss = F.mse_loss(recon_x, x, reduction='mean')
                    kl_divergence = -0.5 * torch.mean(1 + 2 * logstd - mu.pow(2) - (2 * logstd).exp())
                    return recon_loss + kl_divergence

                # Training loop
                for epoch in range(1, epochs + 1):
                    model.train()
                    optimizer.zero_grad()
                    
                    total_loss = 0
                    for data in all_data_list:
                        data = data.to(device)
                        x = data.x.to(device)
                        edge_index = data.edge_index.to(device)
                        mu, logstd = encoder(data.x, data.edge_index)
                        z = mu + torch.exp(logstd) * torch.randn_like(mu)
                        recon = decoder(z.to(device))  # Ensure z is on the same device as the decoder
                        loss = loss_fn(recon, x, mu, logstd)
                        loss.backward()
                        optimizer.step()
                        total_loss += loss.item()

                    avg_loss = total_loss / len(all_data_list)
                    if epoch % 10 == 0:
                        print(f"Epoch {epoch}/{epochs}, Loss: {avg_loss:.4f}")

                # Update best configuration if current loss is lower
                if avg_loss < best_loss:
                    best_loss = avg_loss
                    best_config = {
                        "hidden_channels": hidden_channels,
                        "learning_rate": learning_rate,
                        "weight_decay": weight_decay,
                        "epochs": epochs
                    }

print("Best hyperparameters found were: ", best_config)

# Train final model with best hyperparameters
hidden_channels = best_config["hidden_channels"]
learning_rate = best_config["learning_rate"]
weight_decay = best_config["weight_decay"]
epochs = best_config["epochs"]

encoder = Encoder(in_channels=4, hidden_channels=hidden_channels)
decoder = Decoder(hidden_channels=hidden_channels, out_channels=4)
model = VGAE(encoder).to(device)
optimizer = torch.optim.Adam(list(encoder.parameters()) + list(decoder.parameters()), lr=learning_rate, weight_decay=weight_decay)

for epoch in range(1, epochs + 1):
    model.train()
    optimizer.zero_grad()
    
    total_loss = 0
    for data in all_data_list:
        data = data.to(device)
        mu, logstd = encoder(data.x, data.edge_index)
        z = model.reparameterize(mu, logstd)
        recon = decoder(z)  # Use the decoder to reconstruct the input
        loss = loss_fn(recon, data.x, mu, logstd)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()

    avg_loss = total_loss / len(all_data_list)
    if epoch % 10 == 0:
        print(f"Epoch {epoch}/{epochs}, Loss: {avg_loss:.4f}")

# Save the trained model
torch.save(model.state_dict(), f'/home/nishioka/GNN/GNNmodel/gvae_model_{timestamp}.pth')

# Plot training loss (for simplicity, a placeholder example)
plt.figure()
plt.plot(range(1, epochs + 1), [avg_loss] * epochs)  # Placeholder; replace with actual epoch losses
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('GVAE Training Loss')
plt.savefig(f'/home/nishioka/GNN/GNNmodelgraphfig/gvae_loss_{timestamp}.png')
plt.show()

GPU is available
Using device: NVIDIA GeForce RTX 3090
Training with hidden_channels=128, learning_rate=0.0001, weight_decay=1e-05, epochs=300
Epoch 10/300, Loss: 0.0589
Epoch 20/300, Loss: 0.0577
Epoch 30/300, Loss: 0.0504
Epoch 40/300, Loss: 0.0468
Epoch 50/300, Loss: 0.0449
Epoch 60/300, Loss: 0.0463
Epoch 70/300, Loss: 0.0469
Epoch 80/300, Loss: 0.0473
Epoch 90/300, Loss: 0.0462
Epoch 100/300, Loss: 0.0472
Epoch 110/300, Loss: 0.0474
Epoch 120/300, Loss: 0.0471
Epoch 130/300, Loss: 0.0473
Epoch 140/300, Loss: 0.0445
Epoch 150/300, Loss: 0.0446
Epoch 160/300, Loss: 0.0470
Epoch 170/300, Loss: 0.0471
Epoch 180/300, Loss: 0.0444
Epoch 190/300, Loss: 0.0452
Epoch 200/300, Loss: 0.0459
Epoch 210/300, Loss: 0.0458
Epoch 220/300, Loss: 0.0472
Epoch 230/300, Loss: 0.0457
Epoch 240/300, Loss: 0.0451
Epoch 250/300, Loss: 0.0461
Epoch 260/300, Loss: 0.0449
Epoch 270/300, Loss: 0.0455
Epoch 280/300, Loss: 0.0457
Epoch 290/300, Loss: 0.0456
Epoch 300/300, Loss: 0.0440
Best hyperparameters found

AttributeError: 'VGAE' object has no attribute 'reparameterize'